# Assignment 02: Object Detection

**Available:** Aug 26, 2025 until Sep 4, 2025 11:59pm

## Tasks:

1. Object Detection- refer to slides 8/26/2025 class
2. https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
3. Run detection performance comparing (inference only):
    - Faster R-CNN
    - DETR
    - DINO
    - Grounding DINO (use text prompt at your discretion)
4. Grading Criteria:
    - Report
    - Code
    - Video
    - Insight


## Import Packages and Setup

In [1]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import torch
import torch.nn as nn
import torchvision
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline

# Install einops for tensor manipulation
%pip install einops

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    # Install huggingface_hub if not already installed
    %pip install huggingface_hub
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"✅ Successfully authenticated with Hugging Face!")
        print(f"👤 Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None

# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Successfully authenticated with Hugging Face!
👤 Logged in as: malneyugnfl

=== GPU Diagnostics ===
PyTorch version: 2.8.0
CUDA available: False
CUDA version: None
Number of GPUs detected: 0
Using CPU
Selected device: cpu


## Import and Process Data

## Import Faster R-CNN

In [ ]:
# Load Faster R-CNN model
from torchvision.models import detection
import torchvision.transforms as T

# Load pre-trained Faster R-CNN model with ResNet-50 backbone
faster_rcnn = detection.fasterrcnn_resnet50_fpn(pretrained=True)
faster_rcnn.eval()  # Set to evaluation mode
faster_rcnn = faster_rcnn.to(device)


## Import DinoV3 Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vit7b16-pretrain-lvd1689m")
model = AutoModel.from_pretrained("facebook/dinov3-vit7b16-pretrain-lvd1689m")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m.
403 Client Error. (Request ID: Root=1-68b7b444-0f5a5c517182062f40ef5702;108e3d64-1e22-4043-96f8-f088b75d5fb0)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m/resolve/b80367753773648a6793235ab9c65cdbb029506f/preprocessor_config.json.
Your request to access model facebook/dinov3-vit7b16-pretrain-lvd1689m is awaiting a review from the repo authors.

## Import DETR Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/facebook/detr-resnet-50
from transformers import AutoImageProcessor, AutoModelForObjectDetection

processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")

## Import Grounding-DINO Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/IDEA-Research/grounding-dino-base
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base")